# Assignment 3: CNN-LSTM, ViT, and Object Tracking
## Section 2: Multi-Object Tracking System
### Prasanna Paithankar (21CS30065)

6th April 2026

In [1]:
!pip install ultralytics
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
class KalmanFilter:
    def __init__(self, bbox):
        cx = (bbox[0] + bbox[2]) / 2.0
        cy = (bbox[1] + bbox[3]) / 2.0
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]

        self.x = np.array([[cx, cy, w, h, 0, 0]]).T

        self.F = np.array(
            [
                [1, 0, 0, 0, 1, 0],
                [0, 1, 0, 0, 0, 1],
                [0, 0, 1, 0, 0, 0],
                [0, 0, 0, 1, 0, 0],
                [0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 1],
            ],
            dtype=np.float32,
        )

        self.H = np.array(
            [
                [1, 0, 0, 0, 0, 0],
                [0, 1, 0, 0, 0, 0],
                [0, 0, 1, 0, 0, 0],
                [0, 0, 0, 1, 0, 0],
            ],
            dtype=np.float32,
        )

        self.P = np.eye(6, dtype=np.float32) * 10.0
        self.P[4:, 4:] *= 100.0
        self.R = np.eye(4, dtype=np.float32) * 1.0
        self.Q = np.eye(6, dtype=np.float32) * 0.1

    def predict(self):
        self.x = np.dot(self.F, self.x)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.get_state()

    def update(self, bbox):
        cx = (bbox[0] + bbox[2]) / 2.0
        cy = (bbox[1] + bbox[3]) / 2.0
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        z = np.array([[cx, cy, w, h]]).T

        y = z - np.dot(self.H, self.x)

        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R

        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))

        self.x = self.x + np.dot(K, y)

        I = np.eye(self.P.shape[0])
        self.P = np.dot((I - np.dot(K, self.H)), self.P)

    def get_state(self):
        cx, cy, w, h = self.x[0:4, 0]
        x1 = cx - w / 2.0
        y1 = cy - h / 2.0
        x2 = cx + w / 2.0
        y2 = cy + h / 2.0
        return np.array([x1, y1, x2, y2])

In [3]:
class Track:
    _id_counter = 1

    def __init__(self, bbox):
        self.id = Track._id_counter
        Track._id_counter += 1
        self.kf = KalmanFilter(bbox)
        self.age = 1
        self.hits = 1
        self.time_since_update = 0
        self.bbox = bbox


def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou


class Tracker:
    def __init__(self, iou_thresh=0.3, max_age=5, min_hits=3):
        self.tracks = []
        self.iou_thresh = iou_thresh
        self.max_age = max_age
        self.min_hits = min_hits

    def update(self, detections):
        for track in self.tracks:
            track.predict_bbox = track.kf.predict()
            track.age += 1
            track.time_since_update += 1

        matched, unmatched_dets, unmatched_trks = [], [], []

        if len(self.tracks) == 0:
            unmatched_dets = list(range(len(detections)))
        else:
            cost_matrix = np.zeros(
                (len(detections), len(self.tracks)), dtype=np.float32
            )
            for d, det in enumerate(detections):
                for t, trk in enumerate(self.tracks):
                    iou = calculate_iou(det, trk.predict_bbox)
                    cost_matrix[d, t] = 1.0 - iou

            row_ind, col_ind = linear_sum_assignment(cost_matrix)

            for d, t in zip(row_ind, col_ind):
                if cost_matrix[d, t] > (1.0 - self.iou_thresh):
                    unmatched_dets.append(d)
                    unmatched_trks.append(t)
                else:
                    matched.append((d, t))

            for d in range(len(detections)):
                if d not in row_ind:
                    unmatched_dets.append(d)

            for t in range(len(self.tracks)):
                if t not in col_ind:
                    unmatched_trks.append(t)

        for d, t in matched:
            track = self.tracks[t]
            track.kf.update(detections[d])
            track.bbox = track.kf.get_state()
            track.hits += 1
            track.time_since_update = 0

        for d in unmatched_dets:
            self.tracks.append(Track(detections[d]))

        active_tracks = []
        valid_tracks = []
        for track in self.tracks:
            if track.time_since_update > self.max_age:
                continue
            active_tracks.append(track)

            if track.hits >= self.min_hits or track.age >= self.max_age:
                valid_tracks.append(track)

        self.tracks = active_tracks
        return valid_tracks

In [4]:
def process_video(input_path, output_path, config):
    model = YOLO("yolov8n.pt")

    tracker = Tracker(
        iou_thresh=config["iou_tau"],
        max_age=config["max_age"],
        min_hits=config["min_hits"],
    )

    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    out = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"XVID"), fps, (width, height)
    )

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        results = model(
            frame,
            classes=[0],
            conf=config["conf_thresh"],
            iou=config["nms_thresh"],
            verbose=False,
        )

        detections = []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            for box in boxes:
                detections.append(box)

        confirmed_tracks = tracker.update(detections)

        for track in confirmed_tracks:
            if track.time_since_update <= 1:
                x1, y1, x2, y2 = map(int, track.bbox)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame,
                    f"ID: {track.id}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2,
                )

        out.write(frame)
        print(f"Processed frame {frame_count}")

    cap.release()
    out.release()
    print(f"Video saved to {output_path}")

In [5]:
final_config = {
    "iou_tau": 0.3,
    "max_age": 5,
    "min_hits": 3,
    "conf_thresh": 0.5,
    "nms_thresh": 0.4,
}

print("Starting Phase 1: Processing Training Video...")
process_video(
    "/kaggle/input/datasets/prasannapaithankar/section-3/tracking_train.avi",
    "train_output.avi",
    final_config,
)

print("\nStarting Phase 2: Processing Testing Video...")
process_video(
    "/kaggle/input/datasets/prasannapaithankar/section-3/tracking_test.avi",
    "test_output.avi",
    final_config,
)

Starting Phase 1: Processing Training Video...
Processed frame 1
Processed frame 2
Processed frame 3
Processed frame 4
Processed frame 5
Processed frame 6
Processed frame 7
Processed frame 8
Processed frame 9
Processed frame 10
Processed frame 11
Processed frame 12
Processed frame 13
Processed frame 14
Processed frame 15
Processed frame 16
Processed frame 17
Processed frame 18
Processed frame 19
Processed frame 20
Processed frame 21
Processed frame 22
Processed frame 23
Processed frame 24
Processed frame 25
Processed frame 26
Processed frame 27
Processed frame 28
Processed frame 29
Processed frame 30
Processed frame 31
Processed frame 32
Processed frame 33
Processed frame 34
Processed frame 35
Processed frame 36
Processed frame 37
Processed frame 38
Processed frame 39
Processed frame 40
Processed frame 41
Processed frame 42
Processed frame 43
Processed frame 44
Processed frame 45
Processed frame 46
Processed frame 47
Processed frame 48
Processed frame 49
Processed frame 50
Processed fr

***